# Miami Open – Odds Data EDA

**Goal:** Understand the raw JSON structure returned by The-Odds-API for `tennis_atp_miami_open` before designing the ETL pipeline.

**Source:** `data/raw/tennis_odds_<timestamp>.json` (extracted locally via `src/ingestion/extract_odds.py`)

**Questions to answer:**
1. What does the top-level structure look like?
2. How are bookmakers, markets, and outcomes nested?
3. What fields are present, and what are their types?
4. Are there nulls or inconsistencies to handle?
5. What does the flattened schema look like?

In [1]:
import json
from pathlib import Path

import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 80)

## 1. Load raw JSON
Pick the most recent file from `data/raw/`.

In [2]:
raw_dir = Path('../data/raw')
latest_file = sorted(raw_dir.glob('tennis_odds_*.json'))[-1]
print(f'Loading: {latest_file}')

with open(latest_file, encoding='utf-8') as f:
    raw = json.load(f)

print(f'Total events: {len(raw)}')

Loading: ..\data\raw\tennis_odds_20260322_025143.json
Total events: 12


## 2. Top-level structure
Inspect the keys of a single event and the overall shape.

In [3]:
# Keys in a single event
sample_event = raw[0]
print('Top-level keys:', list(sample_event.keys()))

Top-level keys: ['id', 'sport_key', 'sport_title', 'commence_time', 'home_team', 'away_team', 'bookmakers']


In [4]:
# Quick DataFrame of top-level scalar fields (no nesting)
scalar_fields = ['id', 'sport_key', 'sport_title', 'commence_time', 'home_team', 'away_team']
df_events = pd.DataFrame([{k: e[k] for k in scalar_fields if k in e} for e in raw])
df_events.head(10)

,id,sport_key,sport_title,commence_time,home_team,away_team
0,a406ef20d83ba17b6cd7590244dd9348,tennis_atp_miami_open,ATP Miami Open,2026-03-22T00:19:00Z,Brandon Nakashima,Marin Cilic
1,ed46c754b02edf1052373eaf1e2f912a,tennis_atp_miami_open,ATP Miami Open,2026-03-22T14:00:00Z,Tommy Paul,Raphael Collignon
2,1d4bc5d45b7681483bf98ad085b50e51,tennis_atp_miami_open,ATP Miami Open,2026-03-22T15:00:00Z,Ethan Quinn,Jiri Lehecka
3,06243765b836c39ed1abff2fbcff3c54,tennis_atp_miami_open,ATP Miami Open,2026-03-22T16:30:00Z,Rafael Jodar,Tomas Martin Etcheverry
4,e3ee321c25d4cc3f82b57f08ad2e2b94,tennis_atp_miami_open,ATP Miami Open,2026-03-22T17:30:00Z,Reilly Opelka,Taylor Fritz
5,cbd2c019a24e53f4e3e279a4c71a1d55,tennis_atp_miami_open,ATP Miami Open,2026-03-22T19:00:00Z,Carlos Alcaraz,Sebastian Korda
6,5fff53152cd97983a4b8ac6fa62f5b28,tennis_atp_miami_open,ATP Miami Open,2026-03-22T20:00:00Z,Martin Landaluce,Karen Khachanov
7,306ec867cbd3d464c632c5c42001e462,tennis_atp_miami_open,ATP Miami Open,2026-03-22T23:00:00Z,Matteo Berrettini,Valentin Vacherot
8,6a4be4a7cc6113f4d5d6443c49b434c0,tennis_atp_miami_open,ATP Miami Open,2026-03-23T00:30:00Z,Arthur Fils,Stefanos Tsitsipas
9,d2962e29ac66ae8c41d9b3eae9942d5e,tennis_atp_miami_open,ATP Miami Open,2026-03-23T15:00:00Z,Corentin Moutet,Jannik Sinner


## 3. Nested structure: bookmakers → markets → outcomes

In [5]:
# How many bookmakers per event on average?
bm_counts = [len(e.get('bookmakers', [])) for e in raw]
print(f'Bookmakers per event — min: {min(bm_counts)}, max: {max(bm_counts)}, avg: {sum(bm_counts)/len(bm_counts):.1f}')

Bookmakers per event — min: 3, max: 7, avg: 5.8


In [6]:
# Inspect one bookmaker entry
sample_bookmaker = sample_event['bookmakers'][0]
print(json.dumps(sample_bookmaker, indent=2))

{
  "key": "betrivers",
  "title": "BetRivers",
  "last_update": "2026-03-22T02:45:03Z",
  "markets": [
    {
      "key": "h2h",
      "last_update": "2026-03-22T02:45:03Z",
      "outcomes": [
        {
          "name": "Brandon Nakashima",
          "price": 2.8
        },
        {
          "name": "Marin Cilic",
          "price": 1.42
        }
      ]
    }
  ]
}


In [7]:
# Inspect one market and its outcomes
sample_market = sample_bookmaker['markets'][0]
print(json.dumps(sample_market, indent=2))

{
  "key": "h2h",
  "last_update": "2026-03-22T02:45:03Z",
  "outcomes": [
    {
      "name": "Brandon Nakashima",
      "price": 2.8
    },
    {
      "name": "Marin Cilic",
      "price": 1.42
    }
  ]
}


## 4. Flatten the nested structure
Each row = one outcome (player price) from one bookmaker for one event.

In [8]:
rows = []
for event in raw:
    event_id = event['id']
    commence_time = event['commence_time']
    home_team = event['home_team']
    away_team = event['away_team']

    for bookmaker in event.get('bookmakers', []):
        bookmaker_key = bookmaker['key']
        bookmaker_title = bookmaker['title']
        last_update = bookmaker.get('last_update')

        for market in bookmaker.get('markets', []):
            market_key = market['key']

            for outcome in market.get('outcomes', []):
                rows.append({
                    'event_id': event_id,
                    'commence_time': commence_time,
                    'home_team': home_team,
                    'away_team': away_team,
                    'bookmaker_key': bookmaker_key,
                    'bookmaker_title': bookmaker_title,
                    'last_update': last_update,
                    'market_key': market_key,
                    'outcome_name': outcome['name'],
                    'price': outcome['price'],
                })

df = pd.DataFrame(rows)
print(f'Shape: {df.shape}')
df.head(20)

Shape: (138, 10)


,event_id,commence_time,home_team,away_team,bookmaker_key,bookmaker_title,last_update,market_key,outcome_name,price
0,a406ef20d83ba17b6cd7590244dd9348,2026-03-22T00:19:00Z,Brandon Nakashima,Marin Cilic,betrivers,BetRivers,2026-03-22T02:45:03Z,h2h,Brandon Nakashima,2.80
1,a406ef20d83ba17b6cd7590244dd9348,2026-03-22T00:19:00Z,Brandon Nakashima,Marin Cilic,betrivers,BetRivers,2026-03-22T02:45:03Z,h2h,Marin Cilic,1.42
2,a406ef20d83ba17b6cd7590244dd9348,2026-03-22T00:19:00Z,Brandon Nakashima,Marin Cilic,draftkings,DraftKings,2026-03-22T02:44:14Z,h2h,Brandon Nakashima,2.87
3,a406ef20d83ba17b6cd7590244dd9348,2026-03-22T00:19:00Z,Brandon Nakashima,Marin Cilic,draftkings,DraftKings,2026-03-22T02:44:14Z,h2h,Marin Cilic,1.41
4,a406ef20d83ba17b6cd7590244dd9348,2026-03-22T00:19:00Z,Brandon Nakashima,Marin Cilic,fanduel,FanDuel,2026-03-22T02:45:04Z,h2h,Brandon Nakashima,2.30
5,a406ef20d83ba17b6cd7590244dd9348,2026-03-22T00:19:00Z,Brandon Nakashima,Marin Cilic,fanduel,FanDuel,2026-03-22T02:45:04Z,h2h,Marin Cilic,1.62
6,ed46c754b02edf1052373eaf1e2f912a,2026-03-22T14:00:00Z,Tommy Paul,Raphael Collignon,draftkings,DraftKings,2026-03-22T02:50:54Z,h2h,Raphael Collignon,3.17
7,ed46c754b02edf1052373eaf1e2f912a,2026-03-22T14:00:00Z,Tommy Paul,Raphael Collignon,draftkings,DraftKings,2026-03-22T02:50:54Z,h2h,Tommy Paul,1.37
8,ed46c754b02edf1052373eaf1e2f912a,2026-03-22T14:00:00Z,Tommy Paul,Raphael Collignon,bovada,Bovada,2026-03-22T02:50:20Z,h2h,Raphael Collignon,3.00
9,ed46c754b02edf1052373eaf1e2f912a,2026-03-22T14:00:00Z,Tommy Paul,Raphael Collignon,bovada,Bovada,2026-03-22T02:50:20Z,h2h,Tommy Paul,1.40


## 5. Data quality checks

In [9]:
print('Dtypes:')
print(df.dtypes)
print('\nNull counts:')
print(df.isnull().sum())

Dtypes:
event_id            object
commence_time       object
home_team           object
away_team           object
bookmaker_key       object
bookmaker_title     object
last_update         object
market_key          object
outcome_name        object
price              float64
dtype: object

Null counts:
event_id           0
commence_time      0
home_team          0
away_team          0
bookmaker_key      0
bookmaker_title    0
last_update        0
market_key         0
outcome_name       0
price              0
dtype: int64


In [10]:
# Unique bookmakers present
print('Bookmakers:', df['bookmaker_key'].unique())

# Price range sanity check
print('\nPrice stats:')
print(df['price'].describe())

Bookmakers: ['betrivers' 'draftkings' 'fanduel' 'bovada' 'lowvig' 'betonlineag'
 'betus']

Price stats:
count    138.000000
mean       2.722609
std        3.140329
min        1.010000
25%        1.402500
50%        1.905000
75%        2.967500
max       23.000000
Name: price, dtype: float64


## 6. Notes & schema decisions

> Fill this in after running the cells above. Document:
> - Final agreed column names and types
> - Any nulls or anomalies found
> - Transform decisions to carry into `src/processing/transform.py`
> - Then update `docs/MIAMI_OPEN_SCHEMA.md` accordingly.